# Capítulo 2 — Probabilidade e Distribuições

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma breve explicação; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 2.1 — O que é Probabilidade

A definição frequentista promete algo verificável: repita um procedimento muitas vezes, e a proporção de sucessos deve se aproximar da probabilidade "verdadeira". Vamos simular exatamente isso — lançar uma moeda justa (P(cara) = 0,5) milhares de vezes e acompanhar a proporção acumulada de caras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)
lancamentos = rng.integers(0, 2, 10000)          # 0 ou 1, moeda justa
proporcao = np.cumsum(lancamentos) / np.arange(1, 10001)

for n in [10, 100, 1000, 10000]:
    print(f"Após {n:>5} lançamentos: proporção de caras = {num(proporcao[n-1], 3)}")

Com 10 lançamentos a proporção pula longe de 0,5 (0,400) — a amostra é pequena demais para o "longo prazo" fazer efeito. Com 100 ela oscila para o outro lado (0,520). É só a partir de 1.000 (0,509), e sobretudo de 10.000 (0,494), que ela gruda perto de 0,5.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 10001), proporcao, color="#2c3e50", linewidth=1)
ax.axhline(0.5, color="#c0392b", linestyle="--", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Número de lançamentos (escala log)")
ax.set_ylabel("Proporção de caras")
plt.tight_layout()
plt.show()

## 2.2 — Regras de Probabilidade

$$P(\bar{A}) = 1 - P(A)$$

In [ ]:
from formato import num

p_falha = 0.02
p_sucesso = 1 - p_falha
print(f"P(requisição falhar)     : {num(p_falha, 2)}")
print(f"P(requisição não falhar) : {num(p_sucesso, 2)}")

Um exemplo comum em infraestrutura de software: dois serviços independentes, cada um no ar 99% do tempo. "Independentes" aqui quer dizer que a queda de um não afeta a chance de queda do outro — não compartilham servidor, banco de dados ou dependência de rede. A probabilidade de **ambos** estarem no ar ao mesmo tempo é o produto:

In [ ]:
from formato import num
p = 0.99
print(f"Ambos os serviços no ar:      {num(p**2, 4)}")
print(f"Pelo menos um fora do ar:     {num(1 - p**2, 4)}")

Dois lançamentos independentes de uma moeda justa bastam para ver isso acontecer. Seja $X$ o número de caras. Cada um dos quatro resultados (cara-cara, cara-coroa, coroa-cara, coroa-coroa) tem probabilidade $0{,}5 \times 0{,}5 = 0{,}25$ pela regra da multiplicação, porque os dois lançamentos são independentes. "Cara-coroa" e "coroa-cara" são mutuamente exclusivos e ambos dão $X = 1$, então a regra da adição soma as duas:

In [ ]:
from formato import num

p_caras = {
    0: 0.5**2,           # coroa-coroa
    1: 2 * 0.5**2,       # cara-coroa OU coroa-cara
    2: 0.5**2,           # cara-cara
}
for k, prob in p_caras.items():
    print(f"P(X = {k}) = {num(prob, 2)}")

## 2.3 — Distribuição Binomial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

É o modelo do lançamento de moeda repetido, do controle de qualidade ("quantas peças defeituosas em 100?") e, sobretudo, do **teste A/B**: quantas conversões entre *n* visitantes de uma página. Sempre que a pergunta for "de *n* tentativas de tudo-ou-nada, quantas deram sucesso?", a resposta segue uma binomial. Seu centro fica em $np$ (a média) e seu espalhamento é $\sqrt{np(1-p)}$ (o desvio-padrão).

In [ ]:
k = np.arange(0, 11)
fig, ax = plt.subplots()
ax.bar(k, stats.binom.pmf(k, 20, 0.1), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de sucessos")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

Duas perguntas diferentes cabem à binomial, e a scipy tem uma função para cada. A probabilidade de **exatamente** *k* sucessos é a `pmf` (função de massa de probabilidade); a de **até** *k* sucessos (isto é, *k* ou menos) é a `cdf` (função de distribuição acumulada).

In [ ]:
print(f"P(exatamente 2 sucessos em 5, p=0,1): {num(stats.binom.pmf(2, 5, 0.1), 4)}")
print(f"P(até 2 sucessos em 5, p=0,1):        {num(stats.binom.cdf(2, 5, 0.1), 4)}")

## 2.4 — Distribuição de Poisson e Relacionadas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

A **Poisson** modela **eventos raros no tempo ou no espaço** — conta quantas vezes algo acontece num intervalo fixo, dada uma taxa média $\lambda$ (a letra grega *lambda*): chegadas de requisições a um servidor por segundo, ligações a um call center por minuto, defeitos por lote, acidentes por trecho de estrada. A binomial contava sucessos num número fixo de tentativas; a Poisson conta ocorrências num intervalo contínuo, sem um número fixo de "tentativas".

In [ ]:
k = np.arange(0, 11)
fig, ax = plt.subplots()
ax.bar(k, stats.poisson.pmf(k, 2), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de eventos no intervalo")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

Como a binomial, a Poisson é **discreta** — conta eventos inteiros —, então o gráfico é de barras. O pico fica em torno de $\lambda = 2$, a taxa média, que é ao mesmo tempo a média **e** a variância da distribuição.

In [ ]:
print(f"Poisson(λ=2): P(nenhum evento) = {num(stats.poisson.pmf(0, 2), 3)}")
print(f"Poisson(λ=2): P(5 ou mais)     = {num(stats.poisson.sf(4, 2), 3)}")

A Poisson conta **quantos** eventos ocorrem num intervalo. Uma pergunta espelhada é: **quanto tempo** se espera até o próximo evento? Essa é descrita pela distribuição **exponencial**, a irmã contínua da Poisson. As duas descrevem o mesmo processo por ângulos diferentes.

In [ ]:
print(f"Se ocorrem 2 eventos por hora, o tempo médio entre eventos é {num(1 / 2 * 60, 0)} minutos.")

## 2.5 — Distribuição Normal

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

A normal é definida por dois números: a **média**, que diz onde está o centro, e o **desvio-padrão**, que diz o quão espalhada ela é. É **simétrica** — a metade à esquerda da média espelha a metade à direita — e tem uma propriedade que vale memorizar, a **regra 68–95–99,7**: cerca de 68% da massa fica a menos de um desvio da média, 95% a menos de dois, e 99,7% a menos de três.

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), color="#2c3e50", linewidth=2)
for k, cor in [(1, "#27ae60"), (2, "#e67e22"), (3, "#c0392b")]:
    ax.axvline(k, color=cor, linestyle="--", alpha=0.6)
    ax.axvline(-k, color=cor, linestyle="--", alpha=0.6)
ax.set_xlabel("Desvios em relação à média")
ax.set_ylabel("Densidade")
plt.tight_layout()
plt.show()

Vamos aplicá-lo às 50.000 rendas de solicitantes de empréstimo do `loans_income` (em dólares), que o Capítulo 3 usará a fundo.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(renda, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da renda")
plt.tight_layout()
plt.show()

O corpo dos pontos segue a reta, mas a ponta direita **sobe** para longe dela: as rendas mais altas são bem mais altas do que uma distribuição normal preveria. É a assinatura da cauda longa à direita da renda — um padrão que o Capítulo 3 retoma ao estudar amostragem — e o número confirma.

In [ ]:
print(f"Assimetria da renda: {num(stats.skew(renda), 2)}")

## 2.6 — Distribuições de Cauda Longa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

O número que mede o peso das caudas é a **curtose**. Uma normal tem curtose zero (na convenção que a scipy usa); uma distribuição de cauda gorda tem curtose alta e positiva. Vamos gerar uma amostra genuinamente de cauda gorda e medi-la.

In [ ]:
amostra = stats.t.rvs(df=3, size=1000, random_state=42)
print(f"Curtose da amostra:      {num(stats.kurtosis(amostra), 1)}")
print(f"Curtose de uma normal:   0")

Curtose … contra zero: essa amostra tem caudas muito mais pesadas que a normal. O QQ-plot mostra isso de forma inconfundível.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(amostra, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da amostra")
plt.tight_layout()
plt.show()

## 2.7 — Distribuição t de Student

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

A t não é uma curiosidade: é a distribuição que aparece toda vez que se estima a média de uma população a partir de uma amostra **pequena**, sem conhecer o desvio-padrão verdadeiro da população. É a distribuição por trás dos intervalos de confiança e testes **clássicos** de média com amostra pequena — a rota tradicional, que o Capítulo 4 usa nos testes t. O Capítulo 3 chega ao intervalo de confiança por um caminho alternativo, o **bootstrap** (seção 3.5), que dispensa a t.

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), color="#2c3e50", linewidth=2, label="Normal")
for gl, cor in [(1, "#c0392b"), (5, "#e67e22")]:
    ax.plot(x, stats.t.pdf(x, gl), color=cor, linewidth=2, linestyle="--", label=f"t({gl})")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

Vale quantificar o quanto as caudas afinam. A tabela abaixo mostra a probabilidade de uma observação cair além de ±2 (dois desvios do centro), para vários graus de liberdade, com a normal na última linha para comparação.

In [ ]:
print(f"{'distribuição':>14}  {'P(|X| > 2)':>10}")
for gl in [1, 5, 30]:
    print(f"{'t(' + str(gl) + ')':>14}  {num(2 * stats.t.sf(2, gl), 3):>10}")
print(f"{'normal':>14}  {num(2 * stats.norm.sf(2), 3):>10}")

## 2.8 — Distribuição Qui-Quadrado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

A **qui-quadrado** é a soma de várias normais-padrão elevadas ao quadrado. Como é uma soma de quadrados, ela só assume valores **positivos** e é **assimétrica à direita**. Sua forma depende de um único parâmetro, os **graus de liberdade** — e uma propriedade elegante cai daí: a **média da qui-quadrado é igual aos seus graus de liberdade**.

In [ ]:
x = np.linspace(0, 20, 200)
fig, ax = plt.subplots()
for gl, cor in [(2, "#27ae60"), (5, "#e67e22"), (10, "#c0392b")]:
    ax.plot(x, stats.chi2.pdf(x, gl), color=cor, linewidth=2, label=f"k = {gl}")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

O uso prático da qui-quadrado é fornecer um **valor crítico**: o ponto acima do qual fica apenas uma pequena fração da distribuição. É contra esse ponto que um teste compara sua estatística.

In [ ]:
print(f"Média de χ²(5):            {num(stats.chi2.mean(5), 0)}  (= graus de liberdade)")
print(f"Valor crítico 5% de χ²(5): {num(stats.chi2.ppf(0.95, 5), 2)}")

## 2.9 — Distribuição F

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Como é uma razão de variâncias — e variância é uma média de quadrados, sempre positiva —, a F também só assume valores **positivos** e é **assimétrica à direita**. Sua forma depende de **dois** parâmetros de graus de liberdade: um do numerador e um do denominador.

In [ ]:
x = np.linspace(0, 5, 200)
fig, ax = plt.subplots()
for (g1, g2), cor in [((5, 10), "#27ae60"), ((10, 30), "#e67e22")]:
    ax.plot(x, stats.f.pdf(x, g1, g2), color=cor, linewidth=2, label=f"F({g1}, {g2})")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

Como a qui-quadrado, a F entrega um **valor crítico**: o ponto acima do qual fica só 5% da distribuição.

In [ ]:
print(f"Valor crítico 5% de F(5, 10): {num(stats.f.ppf(0.95, 5, 10), 2)}")